# Monte Carlo Test Results Analysis

This notebook provides an interactive interface for analyzing Monte Carlo test failures.

## Usage
1. Run the first cell to load the latest failure data
2. Use the provided helper functions to query and filter results
3. Analyze specific failure cases with detailed inspection tools

In [ ]:
"""Enhanced analysis of Monte Carlo test failures with improved querying capabilities."""

import json
from pathlib import Path
from glob import glob
import pandas as pd
import numpy as np

# Configuration
OUTPUT_DIR = Path(".")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("tests/output")

# Find the latest failure reports
files = sorted(glob(str(OUTPUT_DIR / "failures_*.json")), reverse=True)
if not files:
    raise FileNotFoundError(f"No failure reports in {OUTPUT_DIR}")

# Load all failure reports
all_failures = {}
for file_path in files:
    path = Path(file_path)
    with open(path) as f:
        failures = json.load(f)
    
    test_name = path.stem.replace("failures_", "").rsplit("_", 1)[0]
    timestamp = "_".join(path.stem.split("_")[-2:])
    
    if test_name not in all_failures:
        all_failures[test_name] = {"timestamp": timestamp, "data": failures}
    
    print(f"Loaded: {path.name}")

# Create DataFrames for easy querying
dataframes = {}
for test_name, test_data in all_failures.items():
    dfs = []
    for cat, items in test_data["data"].items():
        df = pd.DataFrame(items)
        df["category"] = cat
        df["test"] = test_name
        df["timestamp"] = test_data["timestamp"]
        dfs.append(df)
    
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        dataframes[test_name] = combined_df
        print(f"Created DataFrame for '{test_name}' with {len(combined_df)} failures")

# Combine all tests into one master DataFrame
if dataframes:
    master_df = pd.concat(dataframes.values(), ignore_index=True)
    print(f"\nMaster DataFrame created with {len(master_df)} total failures across all tests")
else:
    master_df = pd.DataFrame()
    print("\nNo failure data available")

print("\nAvailable DataFrames:")
for name in dataframes.keys():
    print(f"  - df_{name.replace('-', '_').replace(' ', '_')}")
print(f"  - master_df (all tests combined)")

# Create global variables for easy access
for name, df in dataframes.items():
    var_name = f"df_{name.replace('-', '_').replace(' ', '_')}"
    exec(f"{var_name} = df")

print("\nSetup complete! Use the helper functions below to analyze failures.")

In [ ]:
"""Helper functions for querying and analyzing Monte Carlo test failures."""

def get_failure_summary(df=None):
    """Get a summary of failures by test and category."""
    if df is None:
        df = master_df
    
    if df.empty:
        print("No failure data available")
        return
    
    summary = df.groupby(['test', 'category']).size().reset_index(name='count')
    summary_pivot = summary.pivot(index='test', columns='category', values='count').fillna(0)
    print("Failure Summary:")
    print(summary_pivot.astype(int))
    
    total_failures = len(df)
    print(f"\nTotal failures: {total_failures}")

def filter_by_heir(df, heir_name):
    """Filter failures that involve a specific heir."""
    return df[df['case'].apply(lambda c: heir_name in c and c[heir_name] not in [0, False])]

def filter_by_status(df, status):
    """Filter failures by calculation status."""
    return df[df['result'].apply(lambda r: r.get('status') == status)]

def filter_by_ending(df, ending):
    """Filter failures by calculation ending."""
    return df[df['result'].apply(lambda r: r.get('ending') == ending)]

def filter_by_total_range(df, min_total=0.99, max_total=1.01):
    """Filter failures by total distribution range."""
    return df[(df['result'].apply(lambda r: r.get('total', 0)) >= min_total) & 
              (df['result'].apply(lambda r: r.get('total', 0)) <= max_total)]

def inspect_case(df, index=None, row=None):
    """Inspect a specific failure case in detail."""
    if row is None:
        if index is None:
            print("Please provide either an index or a row")
            return
        row = df.iloc[index]
    
    print(f"Index: {row['index']}")
    print(f"Category: {row['category']}")
    print(f"Test: {row['test']}")
    print(f"Timestamp: {row['timestamp']}")
    print("\nInput Case:")
    for key, value in row['case'].items():
        print(f"  {key}: {value}")
    
    print("\nResult:")
    result = row['result']
    print(f"  Status: {result.get('status', 'N/A')}")
    print(f"  Ending: {result.get('ending', 'N/A')}")
    print(f"  Total: {result.get('total', 'N/A')}")
    print(f"  Denominator: {result.get('denominator', 'N/A')}")
    
    print("\nDistribution:")
    distribution = result.get('distribution', {})
    for heir, fraction in distribution.items():
        print(f"  {heir}: {fraction}")
    
    print("\nNumerators:")
    numerators = result.get('numerators', {})
    for heir, numerator in numerators.items():
        print(f"  {heir}: {numerator}")

def find_similar_cases(df, case_pattern):
    """Find cases that match a pattern (dictionary of key-value pairs)."""
    def matches_pattern(case, pattern):
        for key, value in pattern.items():
            if key not in case or case[key] != value:
                return False
        return True
    
    return df[df['case'].apply(lambda c: matches_pattern(c, case_pattern))]

def sample_failures(df, n=5):
    """Get a random sample of failures for quick inspection."""
    if len(df) <= n:
        return df
    return df.sample(n=n)

print("Helper functions loaded:")
print("  - get_failure_summary(df): Get summary of failures")
print("  - filter_by_heir(df, heir_name): Filter by specific heir")
print("  - filter_by_status(df, status): Filter by calculation status")
print("  - filter_by_ending(df, ending): Filter by calculation ending")
print("  - filter_by_total_range(df, min_total, max_total): Filter by total distribution range")
print("  - inspect_case(df, index): Inspect a specific case in detail")
print("  - find_similar_cases(df, case_pattern): Find cases matching a pattern")
print("  - sample_failures(df, n): Get random sample of failures")

In [ ]:
"""Usage examples for the analysis functions."""

# Get overall summary
get_failure_summary()

# Example queries (uncomment to run):
# Show failures involving spouses only
# spouse_failures = filter_by_heir(master_df, 'zawj')
# get_failure_summary(spouse_failures)

# Show failures with incomplete status
# incomplete_failures = filter_by_status(master_df, 'Incomplete')
# sample_incomplete = sample_failures(incomplete_failures, 3)
# for i in range(len(sample_incomplete)):
#     inspect_case(sample_incomplete, i)
#     print("-" * 50)

# Show failures with radd ending
# radd_failures = filter_by_ending(master_df, 'radd')
# get_failure_summary(radd_failures)

# Find cases with specific patterns
# zawj_only_cases = find_similar_cases(master_df, {'zawj': True})
# print(f"Found {len(zawj_only_cases)} cases with only zawj")
# if len(zawj_only_cases) > 0:
#     inspect_case(zawj_only_cases, 0)

In [ ]:
"""Detailed inspection of specific failure cases."""

# Inspect the first failure in detail
if not master_df.empty:
    print("=== First Failure Case ===")
    inspect_case(master_df, 0)
else:
    print("No failures to inspect")